# Classes and inheritance

The answers to Section 5 of the sample questions, `documentation/tex/questions`, run.
The questions concern Appendix B of the lecture notes, on the five books of Section 1.2.4,
the abstract class of the frames and the state of a simulated Hawkes process.

Each question's code is copied here verbatim, wrapped in a function where it prints.
`outcome` returns what a call prints, or the name of the exception it raises.
`verify` evaluates every choice of the question and asserts that the key holds and that no
other choice does.
Prices are integer counts of ticks.

In [ ]:
import contextlib
import io
import timeit

import numpy as np
import pandas as pd
import pandera.pandas as pa
from scipy.linalg import expm

from unito26.lob import config
from unito26.lob.frames import FrameSerializable
from unito26.lob.hawkes import ExponentialHawkes, HawkesParams, _HawkesState
from unito26.lob.messages import (
    BUY, SELL, GridDepth, limit_order, market_order, withdrawal,
)
from unito26.lob.orderbook import (
    AXIS_B_VARIANTS, AggregateBook, CachedBestBook, HeapBook, TickArrayBook,
)


def outcome(function, *arguments):
    """What a call prints, or the name of the exception it raises."""
    printed = io.StringIO()
    try:
        with contextlib.redirect_stdout(printed):
            function(*arguments)
    except Exception as error:
        return type(error).__name__
    return printed.getvalue().strip()


def verify(label, key, holds):
    """Assert that, of the choices of a question, the one lettered `key` holds and no other does."""
    holding = [letter for letter, held in zip("ABCDE", holds) if held]
    assert holding == [key], f"{label}: {holding or 'no choice'} hold, the key is {key}"
    print(f"{label}: ({key}) holds, and no other choice does")

Some snippets print and then raise.
`printed_and_raised` returns both: what a call printed before it returned or raised,
and the name of the exception it raised, or `None`.

In [ ]:
def printed_and_raised(function, *arguments):
    """What a call prints, and the name of the exception it raises or None."""
    printed = io.StringIO()
    raised = None
    try:
        with contextlib.redirect_stdout(printed):
            function(*arguments)
    except Exception as error:
        raised = type(error).__name__
    return printed.getvalue().strip(), raised

## `question.writesToABookAndAMessage`: five assignments and a call

Each statement is run on its own, on a fresh book and a fresh message.

In [ ]:
statements = """
book.best_bid = 100        # (1)
book.spread = 1            # (2)
half = book.spread() / 2   # (3)
book.bids = {100: 50}      # (4)
message.size = 0           # (5)
"""


def run(statement):
    book = AggregateBook.from_levels({99: 100}, {101: 100})
    message = limit_order(1.0, 50, 100, BUY)
    exec(statement)


results = [outcome(run, statement) for statement in statements.strip().split("\n")]
for number, result in enumerate(results, start=1):
    print(f"({number})", result or "no exception")
raised = [result != "" for result in results]
verify("question.writesToABookAndAMessage", "A", [
    raised == [False, True, True, False, True],
    raised == [True, True, True, False, True],
    raised == [False, False, True, False, True],
    raised == [False, True, False, False, True],
    raised == [False, True, True, True, True],
])

## `question.frozenSubmitResultHoldsAList`: a frozen record holding a list

In [ ]:
def run():
    book = AggregateBook.from_levels({99: 100}, {101: 100, 102: 80})
    result = book.apply(market_order(1.0, 150, BUY), record=True)
    result.fills.append(result.fills[0])
    print(result.market_order_size, book.last_traded_size)
    result.fills = result.fills[:1]
    print(result.market_order_size)


printed, raised = printed_and_raised(run)
print(repr(printed), raised)
verify("question.frozenSubmitResultHoldsAList", "C", [
    printed == "" and raised == "FrozenInstanceError",
    printed == "150 150" and raised == "FrozenInstanceError",
    printed == "250 150" and raised == "FrozenInstanceError",
    printed == "250 150\n100" and raised is None,
])

## `question.bandMarginShadowedOnTheInstance`: a class attribute read by a class method

In [ ]:
def run():
    class NarrowBook(TickArrayBook):
        BAND_MARGIN = 0

    prices = [1000, 1010]
    wide = TickArrayBook.for_prices(prices)
    wide.BAND_MARGIN = 0
    again = wide.for_prices(prices)
    narrow = NarrowBook.for_prices(prices)
    print(again.floor, again.width, narrow.floor, narrow.width)


printed = outcome(run)
print(printed)
verify("question.bandMarginShadowedOnTheInstance", "B", [
    printed == "1000 11 1000 11",
    printed == "936 139 1000 11",
    printed == "936 139 936 139",
    printed == "1000 11 936 139",
])

## `question.bypassedWritesOnThreeBooks`: writes past `set_size`, and two indexes

In [ ]:
def run():
    prices = []
    for cls in (AggregateBook, CachedBestBook, HeapBook):
        book = cls.from_levels({99: 100, 98: 50}, {101: 100})
        book.bids[100] = 50
        del book.bids[99]
        prices.append(book.best_bid_price)
    print(prices)


printed = outcome(run)
print(printed)
verify("question.bypassedWritesOnThreeBooks", "D", [
    printed == "[100, 100, 100]",
    printed == "[100, 99, 100]",
    printed == "[100, 99, 99]",
    printed == "[100, 99, 98]",
    printed == "[100, 98, 98]",
])

## `question.lookupsCountedThroughTheInheritedMatching`: an override reached by inherited code

In [ ]:
class CountingBook(AggregateBook):
    def __init__(self, strict):
        super().__init__(strict)
        self.lookups = 0

    def best_price(self, direction):
        self.lookups += 1
        return super().best_price(direction)

In [ ]:
def run():
    book = CountingBook.from_levels({99: 100}, {101: 100, 102: 80, 105: 20})
    stream = [market_order(1.0, 190, BUY), market_order(2.0, 40, BUY)]
    counts = [book.lookups]
    for message in stream:
        book.apply(message, record=False)
        counts.append(book.lookups)
    print(counts, book.best_bid_price)


printed = outcome(run)
print(printed)
verify("question.lookupsCountedThroughTheInheritedMatching", "E", [
    printed == "[0, 3, 5] 105",
    printed == "[0, 0, 0] 105",
    printed == "[2, 5, 6] 105",
    printed == "[2, 5, 7] 99",
    printed == "[2, 5, 7] 105",
])

## `question.rescansCountedThroughTheCache`: an override reached from the intermediate class

The stream empties four levels and moves a best price three times;
the cache is rescanned only where an emptied level was the best one.

In [ ]:
class AuditedBook(CachedBestBook):
    def __init__(self, strict):
        super().__init__(strict)
        self.rescans = 0

    def _rescan(self, direction):
        self.rescans += 1
        return super()._rescan(direction)

In [ ]:
def run():
    book = AuditedBook.from_levels(
        {99: 100, 98: 50}, {101: 100, 102: 80, 105: 20}
    )
    stream = [
        market_order(1.0, 150, BUY),
        withdrawal(2.0, 20, 105, SELL),
        withdrawal(3.0, 50, 98, BUY),
        limit_order(4.0, 40, 100, BUY),
        market_order(5.0, 40, SELL),
    ]
    for message in stream:
        book.apply(message, record=False)
    print(book.rescans)


printed = outcome(run)
print(printed)
verify("question.rescansCountedThroughTheCache", "B", [
    printed == "0",
    printed == "2",
    printed == "3",
    printed == "4",
    printed == "5",
])

## `question.indexedBookWithoutSuper`: an override that does not call the function it overrides

In [ ]:
class IndexedBook(AggregateBook):
    def __init__(self, strict):
        super().__init__(strict)
        self.occupied = {BUY: set(), SELL: set()}

    def best_price(self, direction):
        prices = self.occupied[direction]
        if not prices:
            return None
        return max(prices) if direction == BUY else min(prices)

    def set_size(self, direction, price, size):
        super().set_size(direction, price, size)
        if size > 0:
            self.occupied[direction].add(price)
        else:
            self.occupied[direction].discard(price)

indexed_a = IndexedBook

In [ ]:
class IndexedBook(AggregateBook):
    def __init__(self, strict):
        super().__init__(strict)
        self.occupied = {BUY: set(), SELL: set()}

    def best_price(self, direction):
        prices = self.occupied[direction]
        if not prices:
            return None
        return max(prices) if direction == BUY else min(prices)

    def set_size(self, direction, price, size):
        if size > 0:
            self.occupied[direction].add(price)
        else:
            self.occupied[direction].discard(price)

indexed_b = IndexedBook

In [ ]:
def run(IndexedBook):
    book = IndexedBook.from_levels({99: 100, 98: 50}, {101: 100, 102: 80})
    print(book.spread, book.best_bid_size)
    book.apply(market_order(1.0, 150, BUY), record=False)
    print(book.best_ask_price, book.last_traded_size)


a = outcome(run, indexed_a).split("\n")
b = outcome(run, indexed_b).split("\n")
b_returned = True
print("A:", a, "   B:", b)
verify("question.indexedBookWithoutSuper", "C", [
    a == ["2 100", "102 150"] and b == ["2 100", "102 150"],
    a == ["2 100", "102 150"] and b == ["2 0", "102 150"],
    a == ["2 100", "102 150"] and b == ["2 0", "None 0"],
    a == ["2 100", "102 150"] and not b_returned,
    a == ["2 100", "102 150"] and b == ["AttributeError"],
])

## `question.copyOfABookWithAVenue`: a copy through `type(self)`

In [ ]:
class VenueBook(CachedBestBook):
    def __init__(self, venue, strict):
        super().__init__(strict)
        self.venue = venue

venue_a = VenueBook

In [ ]:
class VenueBook(CachedBestBook):
    def __init__(self, strict, venue="MTA"):
        super().__init__(strict)
        self.venue = venue

venue_b = VenueBook

In [ ]:
def run(VenueBook):
    book = VenueBook(venue="XETRA", strict=False)
    book.set_size(BUY, 99, 100)
    clone = book.copy()
    print(clone.venue, clone.best_bid_price)


a, b = outcome(run, venue_a), outcome(run, venue_b)
print("A:", a, "   B:", b)
try:
    venue_a(venue="XETRA", strict=False).copy()
except TypeError as error:
    print("the error of A:", error)
verify("question.copyOfABookWithAVenue", "A", [
    a == "TypeError" and b == "MTA 99",
    a == "XETRA 99" and b == "XETRA 99",
    a == "TypeError" and b == "XETRA 99",
    a == "False 99" and b == "MTA 99",
    a == "TypeError" and b == "MTA None",
])

## `question.buildingABookOfAGivenClass`: three ways to build a book of a class handed in

In [ ]:
def fill(book, bids, asks):
    for direction, levels in ((BUY, bids), (SELL, asks)):
        for price, size in levels.items():
            book.set_size(direction, price, size)
    return book

def build_a(cls, bids, asks):
    return fill(cls(False), bids, asks)

def build_b(cls, bids, asks):
    prices = list(bids) + list(asks)
    return fill(AggregateBook.for_prices(prices), bids, asks)

def build_c(cls, bids, asks):
    prices = list(bids) + list(asks)
    return fill(cls.for_prices(prices), bids, asks)

In [ ]:
def run():
    bids, asks = {99: 100}, {101: 100}
    for build in (build_a, build_b, build_c):
        names = []
        for cls in (CachedBestBook, TickArrayBook):
            try:
                names.append(type(build(cls, bids, asks)).__name__)
            except TypeError:
                names.append("TypeError")
        print(names)


lines = outcome(run).split("\n")
print(*lines, sep="\n")
try:
    build_a(TickArrayBook, {99: 100}, {101: 100})
except TypeError as error:
    print("the error of build_a:", error)
cached_tick ="['CachedBestBook', 'TickArrayBook']"
cached_error = "['CachedBestBook', 'TypeError']"
aggregate = "['AggregateBook', 'AggregateBook']"
verify("question.buildingABookOfAGivenClass", "D", [
    lines == [cached_tick, cached_tick, cached_tick],
    lines == [cached_error, cached_tick, cached_tick],
    lines == [cached_tick, aggregate, cached_tick],
    lines == [cached_error, aggregate, cached_tick],
    lines == [cached_error, aggregate, cached_error],
])

## `question.fiveBooksOnPricesOutsideTheBand`: substitution within a band

In [ ]:
def run():
    stream = [limit_order(1.0, 10, 166, SELL), limit_order(2.0, 10, 34, BUY)]
    for message in stream:
        refused = []
        for cls in AXIS_B_VARIANTS:
            book = cls.from_levels({99: 100}, {101: 100})
            try:
                book.apply(message, record=False)
            except ValueError:
                refused.append(cls.__name__)
        print(refused)


lines = outcome(run).split("\n")
print(*lines, sep="\n")
both = "['BitmapBook', 'TickArrayBook']"
tick = "['TickArrayBook']"
verify("question.fiveBooksOnPricesOutsideTheBand", "E", [
    lines == ["[]", "[]"],
    lines == [tick, tick],
    lines == [both, both],
    lines == ["[]", both],
    lines == [tick, both],
])

## `question.cachedBestPriceWhenTheTouchEmpties`: where the cache pays

Each pair of messages returns the book to where it was, so one book folds its stream
repeatedly. Each fold is timed five times at each depth, and the least time is kept; the
choices are read off the growth of each time per message from $\ell = 10^3$ to
$\ell = 10^4$, and off the ratios at $\ell = 10^4$.

In [ ]:
def deep_book(cls, depth):
    asks = {1001 + k: 100 for k in range(depth)}
    return cls.from_levels({1000: 100}, asks)

def stream(size):
    messages = []
    for k in range(500):
        messages.append(market_order(2.0 * k, size, BUY))
        messages.append(limit_order(2.0 * k + 1, size, 1001, SELL))
    return messages

In [ ]:
def seconds_per_message(cls, depth, messages):
    book = deep_book(cls, depth)

    def fold():
        for message in messages:
            book.apply(message, record=False)

    return min(timeit.repeat(fold, number=1, repeat=5)) / len(messages)


streams = {"S": stream(10), "E": stream(100)}
timings = {}
for depth in (100, 1_000, 10_000):
    for name, messages in streams.items():
        for cls in (AggregateBook, CachedBestBook):
            timings[cls.__name__, name, depth] = seconds_per_message(cls, depth, messages)
        print(f"l = {depth:>6}, stream {name}:  "
              f"AggregateBook {timings['AggregateBook', name, depth] * 1e6:8.2f} us   "
              f"CachedBestBook {timings['CachedBestBook', name, depth] * 1e6:8.2f} us")


def growth(cls, name):
    return timings[cls, name, 10_000] / timings[cls, name, 1_000]


def plain_over_cached(name):
    return timings["AggregateBook", name, 10_000] / timings["CachedBestBook", name, 10_000]


for name in streams:
    print(f"stream {name}: growth of AggregateBook {growth('AggregateBook', name):.1f}x, "
          f"of CachedBestBook {growth('CachedBestBook', name):.1f}x, "
          f"AggregateBook over CachedBestBook at 10^4 {plain_over_cached(name):.1f}x")
verify("question.cachedBestPriceWhenTheTouchEmpties", "D", [
    growth("AggregateBook", "S") < 3 and growth("CachedBestBook", "S") < 3
    and growth("AggregateBook", "E") > 3 and growth("CachedBestBook", "E") > 3,
    all(growth("CachedBestBook", name) < 3 and growth("AggregateBook", name) > 3
        for name in streams),
    plain_over_cached("S") < 1 and plain_over_cached("E") < 1,
    growth("CachedBestBook", "S") < 3 and growth("AggregateBook", "S") > 3
    and growth("CachedBestBook", "E") > 3 and growth("AggregateBook", "E") > 3,
    all(growth(cls, name) < 3
        for cls in ("AggregateBook", "CachedBestBook") for name in streams),
])

## `question.abstractMethodRaisesAtInstantiation`: two ways of leaving a function to a subclass

In [ ]:
class TouchFrame(FrameSerializable):
    def __init__(self, bid, ask):
        self.bid, self.ask = bid, ask

    @classmethod
    def schema(cls):
        return pa.DataFrameSchema({"Bid": pa.Column(int)})

    def to_frame(self):
        frame = pd.DataFrame({"Bid": [self.bid], "Ask": [self.ask]})
        return self.schema().validate(frame)

In [ ]:
def run_touch():
    book = AggregateBook.from_levels({99: 100}, {101: 100})
    touch = TouchFrame(book.best_bid_price, book.best_ask_price)
    print(touch.to_json(0))


def run_source():
    params = HawkesParams([0.5, 0.5], [[0.2, 0.0], [0.0, 0.2]], 1.0)
    source = _HawkesState(params, 7)
    print(source.total_intensity)
    print(next(source.events(10.0)))


touch = printed_and_raised(run_touch)
source = printed_and_raised(run_source)
at_the_call = outcome(lambda: TouchFrame(99, 101))
created = outcome(lambda: _HawkesState(HawkesParams([0.5, 0.5], [[0.2, 0.0], [0.0, 0.2]], 1.0), 7))
print("TouchFrame:", touch, "  the call of the class alone:", repr(at_the_call))
print("_HawkesState:", source, "  the call of the class alone:", repr(created))
try:
    TouchFrame(99, 101)
except TypeError as error:
    print("the error of TouchFrame:", error)
verify("question.abstractMethodRaisesAtInstantiation", "A", [
    touch == ("", "TypeError") and at_the_call == "TypeError"
    and source == ("1.0", "NotImplementedError") and created == "",
    touch[1] is None and source == ("1.0", "NotImplementedError"),
    touch == ("", "TypeError") and source == ("", "TypeError"),
    touch[1] is None and touch[0] != "" and created == "TypeError" and source == ("", "TypeError"),
])